# 📘 Notebook 08 · 量化必备的数学统计基础

> 这一节是**纵向**穿插所有 notebook 的数学骨架。**不需要按顺序学完再做其他事**——遇到具体公式回头查就行。

**包含 7 个模块：**

1. 概率论：分布、条件、独立、贝叶斯
2. 统计推断：估计、检验、置信区间
3. 线性代数：矩阵分解、SVD、PCA
4. 优化方法：凸优化、KKT、拉格朗日
5. 时间序列：平稳性、ARIMA、GARCH、协整、VAR
6. 随机过程：布朗运动、Itô 引理、SDE
7. 信息论 + 因果推断：熵、KL、互信息、DID/IV/RDD

**用法：**
- 第一次浏览：每节看 markdown，不做代码
- 实战用到时：回到对应小节，做代码
- 面试前：把每节的"必背公式"再过一遍

**预计第一遍 6-8 小时，后续随用随查。**

## 1. 概率论

### 1.1 七个必背分布

| 分布 | PDF / PMF | 期望 | 方差 | 何时遇到 |
|---|---|---|---|---|
| 伯努利 $\text{Ber}(p)$ | $p^x(1-p)^{1-x}$ | $p$ | $p(1-p)$ | 二分类、事件发生 |
| 二项 $B(n,p)$ | $\binom{n}{x}p^x(1-p)^{n-x}$ | $np$ | $np(1-p)$ | $n$ 次伯努利之和 |
| 几何 $\text{Geom}(p)$ | $(1-p)^{x-1}p$ | $1/p$ | $(1-p)/p^2$ | 首次成功前失败次数 |
| 泊松 $\text{Poisson}(\lambda)$ | $\frac{\lambda^x e^{-\lambda}}{x!}$ | $\lambda$ | $\lambda$ | 单位时间事件数（如订单到达）|
| 正态 $N(\mu,\sigma^2)$ | $\frac{1}{\sqrt{2\pi}\sigma}e^{-\frac{(x-\mu)^2}{2\sigma^2}}$ | $\mu$ | $\sigma^2$ | CLT、收益率近似 |
| 对数正态 | $\frac{1}{x\sigma\sqrt{2\pi}}e^{-\frac{(\ln x - \mu)^2}{2\sigma^2}}$ | $e^{\mu+\sigma^2/2}$ | $(e^{\sigma^2}-1)e^{2\mu+\sigma^2}$ | 资产价格（GBM）|
| t 分布 | 复杂 | 0 | $\nu/(\nu-2)$ | 重尾、小样本检验 |

### 1.2 三个最容易考的概念

**条件概率：** $P(A|B) = \frac{P(A \cap B)}{P(B)}$

**全概率公式：** $P(A) = \sum_i P(A|B_i)P(B_i)$

**贝叶斯定理：** $P(A|B) = \frac{P(B|A)P(A)}{P(B)}$

→ 量化中最重要的应用：**贝叶斯更新**。先验信念 + 新证据 = 后验信念。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 画几个核心分布对比
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

x = np.linspace(-5, 5, 200)
axes[0].plot(x, stats.norm.pdf(x, 0, 1), label='N(0,1)', color='steelblue')
axes[0].plot(x, stats.t.pdf(x, df=3), label='t(df=3) 重尾', color='darkred')
axes[0].plot(x, stats.laplace.pdf(x, 0, 1), label='Laplace 更重尾', color='darkgreen')
axes[0].set_title('正态 vs 重尾分布'); axes[0].legend(); axes[0].grid(alpha=0.3)

x2 = np.linspace(0, 20, 200)
for lam in [1, 3, 7]:
    axes[1].plot(x2, stats.poisson.pmf(np.arange(20), lam), '-o', label=f'λ={lam}', alpha=0.7)
axes[1].set_title('泊松分布（不同 λ）'); axes[1].legend(); axes[1].grid(alpha=0.3)

x3 = np.linspace(0.01, 5, 200)
for sigma in [0.3, 0.6, 1.0]:
    axes[2].plot(x3, stats.lognorm.pdf(x3, sigma), label=f'σ={sigma}', alpha=0.7)
axes[2].set_title('对数正态（资产价格分布）'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 1.3 中心极限定理 (CLT) 的实战意义

**陈述：** 独立同分布的随机变量之和（标准化后）→ 正态分布。

$$\frac{\sum_{i=1}^n X_i - n\mu}{\sigma\sqrt{n}} \xrightarrow{d} N(0, 1)$$

**对量化的意义：**

- 日收益率近似正态（不严格，重尾）
- $n$ 笔交易的累计 PnL 越来越接近正态
- 这是大数定律 + Sharpe 比率年化的理论基础

**反直觉应用：** 高频交易者赚的是"中心极限定理钱"——单笔期望小但稳定，累计起来稳定。Long-tail 风险的策略反而难以年化。

In [ ]:
# 演示 CLT：抛骰子的均值随 n 增大变正态
np.random.seed(42)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for idx, n in enumerate([1, 5, 30, 100]):
    means = []
    for _ in range(5000):
        means.append(np.random.randint(1, 7, size=n).mean())
    axes[idx].hist(means, bins=40, density=True, alpha=0.6, color='steelblue')
    axes[idx].set_title(f'n={n} 个骰子均值的分布')
    axes[idx].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('n=1 时是离散均匀分布，n=100 时已经几乎是正态')

## 2. 统计推断

### 2.1 估计

**矩估计（Method of Moments）**：用样本矩等于总体矩列方程

**极大似然估计（MLE）**：找参数 $\theta$ 使 $\mathcal{L}(\theta) = \prod_i f(x_i; \theta)$ 最大

**最小二乘（OLS）**：$\hat\beta = (X^T X)^{-1} X^T y$

→ OLS 和 MLE（误差正态时）等价。

### 2.2 假设检验六步法

1. 设原假设 $H_0$（如：策略期望收益 = 0）
2. 设备择 $H_1$（如：策略期望收益 > 0）
3. 选检验统计量（如 t 统计量）
4. 算 p 值（在 $H_0$ 下观察到这么极端结果的概率）
5. 比较 p 与显著性水平 $\alpha$（通常 0.05）
6. 决策：拒绝/不拒绝 $H_0$

### 2.3 量化最常用的三个检验

| 检验 | 用途 | 拒绝域 |
|---|---|---|
| **t 检验** | 均值是否为 0（如 Sharpe 显著性） | $|t| > t_{α/2, n-1}$ |
| **ADF 检验** | 序列是否平稳 | p < 0.05 → 平稳 |
| **Engle-Granger** | 两序列是否协整 | p < 0.05 → 协整 |

### 2.4 p-hacking 与多重检验

**问题：** 试 100 个策略，挑显著的 5 个上线 → 几乎全是假阳性。

**修正方法：**

- **Bonferroni**：$\alpha / m$（保守）
- **Holm-Bonferroni**：按 p 值排序，逐步严苛
- **BH (FDR)**：控制虚假发现率（更宽松，更现实）
- **White Reality Check / Hansen SPA**：考虑联合分布（金融专用）

In [ ]:
# Bonferroni 与 BH 演示
np.random.seed(7)
m = 100   # 试 100 个策略
# 假设 95 个是真垃圾（pvalue 均匀分布），5 个是真好（pvalue 接近 0）
p_null  = np.random.uniform(0, 1, 95)
p_real  = np.random.uniform(0, 0.01, 5)
pvals = np.concatenate([p_null, p_real])
np.random.shuffle(pvals)

# 朴素：p < 0.05
naive = (pvals < 0.05).sum()

# Bonferroni
bonf = (pvals < 0.05 / m).sum()

# Benjamini-Hochberg
from statsmodels.stats.multitest import multipletests
reject, p_adj, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
bh = reject.sum()

print(f'朴素 (α=0.05)        : 拒绝 {naive} 个 ← 包含很多假阳性')
print(f'Bonferroni (α/m)     : 拒绝 {bonf} 个 ← 太保守可能漏真')
print(f'Benjamini-Hochberg   : 拒绝 {bh} 个 ← 平衡')

## 3. 线性代数

### 3.1 矩阵基础

**核心运算**：
- $A^T A$：Gram 矩阵，对称半正定
- $A^{-1}$：逆，仅方阵 + 满秩才存在
- $\det(A)$：行列式，体积变换比例
- $\text{tr}(A)$：迹，对角和 = 特征值和

**重要恒等式**：
- $(AB)^T = B^T A^T$
- $(AB)^{-1} = B^{-1} A^{-1}$
- $\det(AB) = \det(A)\det(B)$

### 3.2 特征分解

$Av = \lambda v$ → 把矩阵 $A$ 分解为 $A = Q\Lambda Q^{-1}$（$\Lambda$ 是特征值对角阵）

**对称矩阵的特殊性**：所有特征值实数，特征向量正交，$A = Q\Lambda Q^T$

### 3.3 SVD（奇异值分解）

$$A_{m \times n} = U_{m \times m} \Sigma_{m \times n} V^T_{n \times n}$$

- $U, V$ 正交
- $\Sigma$ 对角，奇异值降序

**量化中的用途：**
- **PCA**：用 SVD 分解协方差矩阵，前 K 个主成分降维
- **协方差收缩**：高频协方差矩阵不满秩，SVD 截断重构
- **风险因子模型**（Barra）：把股票收益分解到 30+ 风险因子

### 3.4 PCA 实战

In [ ]:
# 用 PCA 分析股票收益的主成分（系统风险 + 行业 + 风格）
np.random.seed(42)
n_days, n_stocks = 500, 50
# 制造一个有 3 个主成分的收益矩阵
mkt = np.random.randn(n_days) * 0.01
size = np.random.randn(n_days) * 0.005
value = np.random.randn(n_days) * 0.005
loads = np.random.rand(n_stocks, 3)
loads[:, 0] = 0.8 + 0.3 * np.random.rand(n_stocks)   # 全部 β 在 0.8~1.1
returns = np.outer(mkt, loads[:, 0]) + np.outer(size, loads[:, 1]) + np.outer(value, loads[:, 2])
returns += np.random.randn(n_days, n_stocks) * 0.01  # 特质噪声

# PCA
U, S, Vt = np.linalg.svd(returns - returns.mean(axis=0), full_matrices=False)
explained = S**2 / (S**2).sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(range(1, 11), explained[:10] * 100, color='steelblue')
axes[0].set_xlabel('主成分'); axes[0].set_ylabel('方差解释比 (%)')
axes[0].set_title('Scree Plot')
axes[0].grid(alpha=0.3)

axes[1].plot(np.cumsum(explained) * 100, 'o-', color='darkred')
axes[1].axhline(80, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('主成分数'); axes[1].set_ylabel('累计方差解释 (%)')
axes[1].set_title('累计方差解释比'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'前 3 个主成分解释了 {explained[:3].sum()*100:.1f}% 的方差')
print('→ 与生成数据的 3 个真实因子一致')

## 4. 优化方法

### 4.1 凸优化

**凸函数定义**：$f(\lambda x + (1-\lambda)y) \le \lambda f(x) + (1-\lambda)f(y)$

**凸优化的好处**：局部最优 = 全局最优

**量化中的凸问题**：
- 最小方差组合：min $w^T \Sigma w$ s.t. $\sum w = 1, w \ge 0$
- 均值-方差优化：min $-\mu^T w + \frac{\gamma}{2} w^T \Sigma w$
- 风险预算：log-barrier 形式

### 4.2 拉格朗日乘子 + KKT 条件

带约束的优化：

$$\min f(x), \text{ s.t. } g_i(x) \le 0, h_j(x) = 0$$

拉格朗日函数：

$$\mathcal{L}(x, \mu, \lambda) = f(x) + \sum_i \mu_i g_i(x) + \sum_j \lambda_j h_j(x)$$

KKT 条件（最优解必满足）：
- 梯度条件：$\nabla_x \mathcal{L} = 0$
- 约束满足：$g_i(x^*) \le 0, h_j(x^*) = 0$
- 对偶可行：$\mu_i \ge 0$
- 互补松弛：$\mu_i g_i(x^*) = 0$

**直觉**：每个不等式约束 $g_i$ 要么不起作用（$\mu_i = 0$），要么紧约束（$g_i = 0$）。

### 4.3 梯度下降家族

| 方法 | 更新规则 | 用途 |
|---|---|---|
| GD | $\theta \leftarrow \theta - \eta \nabla f$ | 凸问题、batch |
| SGD | 每次只用一个样本 | 大数据 |
| Momentum | 加上历史梯度方向 | 加速收敛 |
| Adam | 适应学习率 + momentum | 深度学习首选 |

**学习率**是最关键的超参数。常见调度：cosine、step decay、warmup + decay。

In [ ]:
# 解一个最小方差组合（用 cvxpy）
try:
    import cvxpy as cp

    np.random.seed(0)
    n = 10
    # 制造一个协方差矩阵
    A = np.random.randn(n, n)
    Sigma = A.T @ A / n + 0.01 * np.eye(n)

    w = cp.Variable(n)
    objective = cp.Minimize(cp.quad_form(w, Sigma))
    constraints = [cp.sum(w) == 1, w >= 0, w <= 0.3]
    cp.Problem(objective, constraints).solve()

    print(f'最小方差组合权重:')
    for i, wi in enumerate(w.value):
        print(f'  asset {i}: {wi:.4f}')
    print(f'\n组合方差: {w.value @ Sigma @ w.value:.6f}')
    print(f'最大单仓: {w.value.max():.4f} (上限 0.3)')
except ImportError:
    print('需要 cvxpy: pip install cvxpy')

## 5. 时间序列分析

### 5.1 平稳性

**严格平稳**：联合分布不随时间平移变化（很少见）

**弱平稳**：均值、方差不变，自相关只依赖滞后

**为什么重要**：所有 ARIMA / 协整 / OLS 的统计推断都要求平稳。**非平稳序列做回归会得到伪回归**（spurious regression）。

**检验**：
- ADF（Augmented Dickey-Fuller）：原假设"有单位根"，p < 0.05 拒绝（即平稳）
- KPSS：原假设"平稳"，p < 0.05 拒绝（即非平稳）

**实战**：股价基本都是非平稳的，**先做对数差分** $r_t = \log P_t - \log P_{t-1}$ 得到收益率，再做后续分析。

### 5.2 ARIMA(p,d,q)

- $p$：自回归阶数 AR(p)
- $d$：差分阶数（让序列平稳）
- $q$：移动平均阶数 MA(q)

**形式：**

$$y_t = c + \sum_{i=1}^p \phi_i y_{t-i} + \sum_{j=1}^q \theta_j \epsilon_{t-j} + \epsilon_t$$

**实战中**：对个股价格预测效果差（信噪比太低），但对**月度宏观数据**（CPI、PPI）还行。

### 5.3 GARCH：波动率建模

**ARCH(p)**：$\sigma_t^2 = \alpha_0 + \sum_{i=1}^p \alpha_i \epsilon_{t-i}^2$

**GARCH(p,q)**：$\sigma_t^2 = \alpha_0 + \sum_{i=1}^p \alpha_i \epsilon_{t-i}^2 + \sum_{j=1}^q \beta_j \sigma_{t-j}^2$

**用途**：
- VaR 计算
- 期权定价（隐含波动率 vs 历史波动率）
- 高频数据的波动率聚集建模

### 5.4 协整（Cointegration）

两个非平稳序列 $X_t, Y_t$，如果存在 $\beta$ 使 $X_t - \beta Y_t$ 平稳，则**协整**。

- 配对交易的理论基础（Notebook 02）
- Engle-Granger 两步法 / Johansen 多变量检验

### 5.5 VAR（向量自回归）

多个序列联合建模：

$$\mathbf{y}_t = c + A_1 \mathbf{y}_{t-1} + \ldots + A_p \mathbf{y}_{t-p} + \boldsymbol{\epsilon}_t$$

**应用**：
- 宏观经济变量联动（GDP、CPI、利率）
- 多资产风险传染（Diebold-Yilmaz spillover）
- 你研究过的"跨市场风险传染"就是 VAR 框架

In [ ]:
# ARIMA + GARCH 演示
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
try:
    from arch import arch_model
    HAS_ARCH = True
except ImportError:
    HAS_ARCH = False
    print('arch 库未装；GARCH 演示跳过：pip install arch')

# 生成有 ARCH 效应的数据（波动率聚集）
np.random.seed(0)
n = 1000
returns = np.zeros(n)
sigma = np.ones(n) * 0.01
for t in range(1, n):
    sigma[t] = np.sqrt(0.000005 + 0.1 * returns[t-1]**2 + 0.85 * sigma[t-1]**2)
    returns[t] = sigma[t] * np.random.randn()

import pandas as pd
ret_s = pd.Series(returns)

# ADF 检验：收益率应该平稳
adf_stat, p, *_ = adfuller(ret_s)
print(f'ADF 检验 p 值: {p:.4f}  →  {"平稳" if p < 0.05 else "非平稳"}')

if HAS_ARCH:
    # GARCH(1,1) 拟合
    am = arch_model(ret_s * 100, vol='GARCH', p=1, q=1)
    res = am.fit(disp='off')
    print(f'\nGARCH 估计:')
    print(f'  ω = {res.params["omega"]:.6f}')
    print(f'  α = {res.params["alpha[1]"]:.4f}')
    print(f'  β = {res.params["beta[1]"]:.4f}  → α+β 接近 1 = 持续波动率冲击')

    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    axes[0].plot(returns, color='steelblue', alpha=0.7); axes[0].set_title('模拟收益率')
    axes[0].grid(alpha=0.3)
    axes[1].plot(sigma, color='darkred'); axes[1].set_title('真实条件波动率（黑），GARCH 估计应该贴近')
    cond_vol = res.conditional_volatility / 100
    axes[1].plot(cond_vol.values, color='orange', alpha=0.7, label='GARCH 估计')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 6. 随机过程：期权定价的语言

### 6.1 布朗运动 (BM)

**性质**：
- $W_0 = 0$
- 增量 $W_t - W_s \sim N(0, t-s)$ 独立
- 路径连续但处处不可导

直觉：每个瞬间都在被一个正态随机变量"踢"一下。

### 6.2 几何布朗运动 (GBM)

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

解：$S_t = S_0 \exp\left[(\mu - \frac{1}{2}\sigma^2)t + \sigma W_t\right]$

→ **资产价格的标准模型**，BS 公式的基础。

### 6.3 Itô 引理（最重要的工具）

设 $X_t$ 满足 $dX_t = a(X_t, t)dt + b(X_t, t)dW_t$，$f(X_t, t)$ 是二次可微函数，则：

$$df = \left(\frac{\partial f}{\partial t} + a\frac{\partial f}{\partial x} + \frac{1}{2}b^2\frac{\partial^2 f}{\partial x^2}\right)dt + b\frac{\partial f}{\partial x}dW_t$$

**关键不同点**：比普通链式法则多一项 $\frac{1}{2}b^2 f_{xx}$（来自 $(dW)^2 = dt$）。

→ 用 Itô 推导 Black-Scholes 方程是金工面试常考。

### 6.4 Black-Scholes 公式

欧式看涨期权价格：

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

**希腊字母**（必背）：
- $\Delta = \partial C/\partial S$ = $N(d_1)$（对冲股票数）
- $\Gamma = \partial^2 C/\partial S^2$（对冲调整速率）
- $\nu$ (Vega) = $\partial C/\partial\sigma$（波动率敏感度）
- $\Theta = \partial C/\partial t$（时间衰减）
- $\rho = \partial C/\partial r$（利率敏感度）

In [ ]:
# Black-Scholes 实现
from scipy.stats import norm

def bs_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def bs_greeks(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega  = S * norm.pdf(d1) * np.sqrt(T)            # 对 1.0 单位 σ
    theta = -(S * norm.pdf(d1) * sigma) / (2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)
    rho   = K * T * np.exp(-r*T) * norm.cdf(d2)
    return delta, gamma, vega, theta, rho

# 一个例子：ATM 期权
S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
price = bs_call(S, K, T, r, sigma)
d, g, v, th, ro = bs_greeks(S, K, T, r, sigma)
print(f'看涨期权价格: ${price:.4f}')
print(f'Delta: {d:.4f}   Γ: {g:.4f}')
print(f'Vega : {v:.4f}    Θ: {th:.4f}/year')
print(f'Rho  : {ro:.4f}')

# 蒙特卡洛验证
np.random.seed(42)
n = 100_000
z = np.random.randn(n)
S_T = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z)
mc_price = np.exp(-r*T) * np.maximum(S_T - K, 0).mean()
print(f'\n蒙特卡洛验证: ${mc_price:.4f}  (应接近 BS 公式)')

## 7. 信息论 + 因果推断

### 7.1 信息论基础

**熵**：$H(X) = -\sum_i p_i \log p_i$，衡量不确定性

**条件熵**：$H(Y|X) = H(X,Y) - H(X)$

**KL 散度**：$D_{KL}(P\|Q) = \sum_i p_i \log\frac{p_i}{q_i}$

- 不对称
- 在变分推断、VAE、强化学习中无处不在

**互信息**：$I(X;Y) = H(X) - H(X|Y)$

- 衡量两个变量的"依赖"（包括非线性，比相关系数强）
- 因子研究中可用 MI 评估"未来收益与因子的依赖"

### 7.2 因果推断（这部分对你尤其重要）

> 量化里有一个长期被误用的词：**"因果"**。预测和因果是两回事。

**因果推断的三大经典方法**（这些你的论文里用过）：

**1. 双重差分 (DID)**：

$$\hat\tau = (\bar y^{\text{treat}}_{\text{post}} - \bar y^{\text{treat}}_{\text{pre}}) - (\bar y^{\text{ctrl}}_{\text{post}} - \bar y^{\text{ctrl}}_{\text{pre}})$$

假设：**平行趋势**（处理组和对照组在没有干预时趋势平行）

**2. 工具变量 (IV)**：

对内生性的回归 $y = \beta x + \epsilon$（其中 $E[x\epsilon] \ne 0$），找一个工具 $z$ 满足：
- 相关性：$\text{cov}(z, x) \ne 0$
- 排他性：$\text{cov}(z, \epsilon) = 0$（z 只通过 x 影响 y）

两阶段：第一阶段 $x = \pi z + u$；第二阶段 $y = \beta \hat x + e$

**3. 断点回归 (RDD)**：

某分配规则在阈值 $c$ 处不连续（如"60 分及格 / 不及格"）。比较阈值附近的处理 vs 对照：

$$\hat\tau = \lim_{x \to c^+} E[y|x] - \lim_{x \to c^-} E[y|x]$$

### 7.3 因果发现 vs 因果推断

| 概念 | 含义 |
|---|---|
| 因果推断 | 已知因果方向，估计效应大小 |
| 因果发现 | 从数据中找出因果方向（PC 算法、NOTEARS）|

最近热门的**因果机器学习**（CausalML、EconML）把 DID/IV/RDD 和 ML 结合，处理高维异质处理效应。

### 7.4 量化中的因果思维

- ❌ "动量因子和未来收益高度相关" ≠ "动量导致收益"
- ✅ 加上经济学逻辑：信息扩散慢 + 投资者从众
- ❌ 训练数据里"高 PE 公司未来表现差" ≠ "高 PE 是预测信号"
- ✅ 因果思维：估值高 → 长期 mean reversion → 未来收益负

**因果思维让你的策略更稳健**：纯统计相关性在分布偏移时崩溃，因果机制更可能保留。

In [ ]:
# DID 实现演示
# 模拟：2020 年起对某政策处理组进行干预，使其收益+0.5%
np.random.seed(42)
n_firms = 200
n_quarters = 20  # 2018Q1 - 2022Q4
treatment_start = 12  # 2021Q1

# 半数公司为处理组
treat = np.random.binomial(1, 0.5, n_firms)

# 面板数据
data = []
for f in range(n_firms):
    base = np.random.randn() * 0.3
    for q in range(n_quarters):
        time_trend = q * 0.01
        treated_now = treat[f] and (q >= treatment_start)
        y = base + time_trend + 0.5 * treated_now + np.random.randn() * 0.2
        data.append({'firm': f, 'quarter': q, 'treat': treat[f],
                     'post': int(q >= treatment_start), 'y': y})
df = pd.DataFrame(data)

# DID 回归：y = β0 + β1·treat + β2·post + β3·(treat*post) + ε
# β3 就是 DID 效应
df['treat_post'] = df['treat'] * df['post']
import statsmodels.formula.api as smf
model = smf.ols('y ~ treat + post + treat_post', data=df).fit()
print('DID 估计:')
print(model.summary().tables[1])
print(f'\n真实处理效应 = 0.5')
print(f'估计处理效应 = {model.params["treat_post"]:.4f}'
      f'（标准误 {model.bse["treat_post"]:.4f}）')

## 8. 一页纸速查表

打印贴在桌前的最小集合：

```
═══ 概率 ═══
P(A|B) = P(A∩B)/P(B)             Bayes: P(A|B) = P(B|A)·P(A)/P(B)
E[X+Y] = E[X] + E[Y]              Var(X+Y) = Var(X) + Var(Y) + 2Cov
CLT: sum/√n → N                   Markov: P(X≥a) ≤ E[X]/a
Chebyshev: P(|X-μ|≥kσ) ≤ 1/k²

═══ 统计 ═══
OLS: β̂ = (X'X)⁻¹X'y
t-stat = (β̂ - β₀) / SE(β̂)        显著性 p < α 拒绝 H₀
MLE: 最大化 log L(θ)               ICIR = mean(IC)/std(IC) · √252

═══ 线性代数 ═══
特征分解: A = QΛQ⁻¹ (一般方阵)
谱分解  : A = QΛQ' (对称)
SVD     : A = UΣV'
PCA     : 协方差/相关阵的特征值分解

═══ 优化 ═══
凸: 局部最优 = 全局最优
KKT: ∇L=0, 约束满足, μ≥0, μ·g=0
Adam: m_t / (1-β₁ᵗ), v_t / (1-β₂ᵗ)

═══ 时序 ═══
平稳: 均值方差不变, ADF p<0.05
ARIMA(p,d,q): p=AR, d=差分, q=MA
GARCH(1,1): σ²_t = ω + αε²_{t-1} + βσ²_{t-1}
协整: X - βY 平稳 (Engle-Granger)

═══ 随机过程 ═══
GBM: dS = μS dt + σS dW
Itô: df = (f_t + af_x + ½b²f_xx)dt + bf_x dW
BS:  C = S·N(d₁) - K·e^(-rT)·N(d₂)
       d₁ = [ln(S/K) + (r+σ²/2)T] / (σ√T)

═══ 信息 / 因果 ═══
熵: H = -Σ p log p                 KL: Σ p log(p/q)
DID: y = β₀ + β₁·D + β₂·post + β₃·D·post
IV:  z↔x, z⊥ε  →  β = cov(y,z)/cov(x,z)
RDD: τ = lim_{x→c⁺} E[y|x] - lim_{x→c⁻} E[y|x]
```

## 9. 下一步

数学统计的内容在所有 notebook 里都会用到。**不要试图一次学完**——遇到了回来查就行。

接下来：

- **Notebook 09**：把这些数学武器装备到深度学习因子挖掘上
- **Notebook 10**：强化学习用于交易决策
- **Notebook 11**：LLM + 另类数据

**学习心态**：数学是工具不是目的。**够用 + 知道在哪查** 远胜过 **想学完再开始**。